In [1]:
import pandas as pd
from dash import Dash, html, dcc
import plotly.graph_objects as go

In [2]:
import plotly.graph_objects as go
import pandas as pd
df = pd.read_csv('results.csv')

In [2]:
ranking_df = df.groupby('Model')['F1'].mean().reset_index()
ranking_df = ranking_df.sort_values(by='F1', ascending=False).reset_index(drop=True)

# 3. กำหนดชุดสีตามที่คุณระบุ (เรียงตามอันดับโมเดล)
_MODEL_COLORS = ["#dc2626", "#0891b2", "#16a34a", "#d97706", "#7c3aed", "#2563eb", "#db2777", "#65a30d"]


# แมตช์สีกับโมเดล (ใช้แค่เท่าที่มีจำนวนโมเดล)
model_colors = _MODEL_COLORS[:len(ranking_df)]

# พลิกข้อมูลเพื่อพล็อต (Plotly จะวาดจากล่างขึ้นบนในแนวนอน)
df_plot = ranking_df.iloc[::-1].reset_index(drop=True)
plot_colors = model_colors[::-1]

# 4. สร้างกราฟ
fig = go.Figure(go.Bar(
    x=df_plot['F1'],
    y=df_plot['Model'],
    orientation='h',
    marker=dict(
        color=plot_colors,
        opacity=0.85, # ปรับความอ่อนเพื่อให้ดูนุ่มนวลขึ้น
        line=dict(color='white', width=1)
    ),
    text=df_plot['F1'].apply(lambda x: f'<b>{x:.2f}</b>'),
    textposition='inside',
    insidetextanchor='end',
    textfont=dict(size=14, color='white'),
    width=0.7 # ความหนาของแท่ง
))

# 5. ปรับแต่ง Layout ให้คล้ายธีมในรูปภาพ
fig.update_layout(
    title=dict(
        text="<b>Model Performance Ranking</b><br><span style='font-size:14px; color:#6b7280;'>Mean F1-score comparison across all models</span>",
        font=dict(size=22, color='#111827'),
        x=0.03,
        y=0.92
    ),
    xaxis=dict(
        title="Mean F1-score (%)",
        showgrid=True,
        gridcolor='#f3f4f6', # เส้น Grid จางๆ
        range=[0, 105],
        dtick=10,
        zeroline=False
    ),
    yaxis=dict(
        title="",
        showgrid=False,
        tickfont=dict(size=14, color='#374151')
    ),
    template="plotly_white",
    margin=dict(l=160, r=40, t=100, b=60),
    height=550,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    bargap=0.2
)

# แสดงผล
fig.show()

In [3]:
import numpy as np
import plotly.graph_objects as go
import random

# --- [FIX: Repeated Division & Out of Bounds Bug] ---
# ใช้ copy เพื่อป้องกันไม่ให้รันซ้ำแล้วค่าโดนหารไปเรื่อยๆ
plot_df = df.copy()

# ตรวจสอบและหาร 100 เฉพาะเมื่อค่าเป็นหลักหน่วย/สิบ (ป้องกันการรันซ้ำใน Cell เดียวกัน)
if plot_df['Precision'].max() > 1.05:
    plot_df['Precision'] = plot_df['Precision'] / 100
if plot_df['Recall'].max() > 1.05:
    plot_df['Recall'] = plot_df['Recall'] / 100

# 1. กำหนดค่าพื้นฐาน
_MODEL_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]
_TH_SYMBOLS = {
    "P99 Static": "circle",
    "Sliding Mu+αStd": "diamond",
    "Adaptive-z": "square",
    "Entropy-lock": "x"
}
_TH_SHORT = {
    "P99 Static": "P99",
    "Sliding Mu+αStd": "Slid",
    "Adaptive-z": "Adpt",
    "Entropy-lock": "Entr"
}

fig = go.Figure()

# 2. F1-Isolines (เส้นโค้งพื้นหลัง)
x_iso = np.linspace(0.01, 1, 100)
for f1 in [0.2, 0.4, 0.6, 0.8]:
    # Precision = (f1 * Recall) / (2 * Recall - f1)
    y_iso = (f1 * x_iso) / (2 * x_iso - f1)
    mask = (y_iso >= 0) & (y_iso <= 1)
    
    rx = x_iso[mask]
    py = y_iso[mask]
    
    if len(rx) > 0:
        fig.add_trace(go.Scatter(
            x=rx, y=py,
            mode="lines",
            line=dict(color="#cbd5e1", width=1, dash="dot"),
            showlegend=False, hoverinfo="skip"
        ))
        
        # [FIX: Dynamic Indexing] เลือกจุดกึ่งกลางของเส้นเพื่อแปะป้าย F1 ป้องกัน IndexError
        label_idx = len(rx) // 2 
        fig.add_annotation(
            x=rx[label_idx], y=py[label_idx],
            text=f"F1={f1}", showarrow=False,
            font=dict(size=9, color="#94a3b8"),
            bgcolor="white", opacity=0.8
        )

# 3. Best Zone Shading & Annotation
fig.add_shape(
    type="rect", x0=0.75, y0=0.75, x1=1.0, y1=1.0,
    fillcolor="rgba(34,197,94,0.06)",
    line=dict(color="rgba(34,197,94,0.25)", width=1, dash="dot"),
    layer="below"
)
fig.add_annotation(
    x=0.875, y=0.88,
    text="✦ Best Zone",
    showarrow=False,
    font=dict(size=10, color="rgba(22,163,74,0.8)", weight="bold"),
    xanchor="center",
    opacity=0.6
)

# 4. วาดจุด (ไม่มี Text ในกราฟตามที่ต้องการ)
models = plot_df['Model'].unique()
for i, model_name in enumerate(models):
    model_data = plot_df[plot_df['Model'] == model_name]
    color = _MODEL_COLORS[i % len(_MODEL_COLORS)]
    
    # วนลูปตาม Threshold เพื่อให้สัญลักษณ์ตรงตามที่กำหนด
    for th_key in _TH_SYMBOLS.keys():
        row = model_data[model_data['Threshold'] == th_key]
        if row.empty: continue
        
        row = row.iloc[0]
        # Jitter เล็กน้อยเพื่อให้เห็นขอบขาวเวลาทับกันเป๊ะๆ
        display_x = row['Recall'] + random.uniform(-0.002, 0.002)
        display_y = row['Precision'] + random.uniform(-0.002, 0.002)
        
        fig.add_trace(go.Scatter(
            x=[display_x],
            y=[display_y],
            mode="markers", # เอา +text ออกแล้ว
            name=f"{model_name} · {_TH_SHORT.get(th_key)}",
            legendgroup=model_name,
            legendgrouptitle=dict(text=model_name) if th_key == "P99 Static" else None,
            marker=dict(
                size=14, 
                color=color, 
                symbol=_TH_SYMBOLS.get(th_key),
                opacity=0.9,
                line=dict(width=1.5, color="white")
            ),
            hovertemplate=(
                f"<b>{model_name}</b><br>"
                f"TH: {th_key}<br>"
                f"Recall: %{{x:.3f}}<br>"
                f"Precision: %{{y:.3f}}<extra></extra>"
            )
        ))

# 5. Layout Setup
fig.update_layout(
    height=550,
    template="plotly_white",
    title=dict(
        text="<b>PR Plot — Precision × Recall</b>",
        font=dict(size=16, color="#0f172a"),
        x=0.02
    ),
    xaxis=dict(
        title="Recall", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    yaxis=dict(
        title="Precision", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    legend=dict(
        tracegroupgap=8, 
        itemsizing='constant',
        font=dict(size=10),
        bordercolor="#e2e8f0", borderwidth=1,
        x=1.02, y=1
    ),
    margin=dict(l=60, r=200, t=80, b=80)
)

# สัญลักษณ์ Legend อ้างอิงด้านล่าง
symbol_hint = " &nbsp;·&nbsp; ".join([f"<b>{v.upper()}</b> = {k}" for k, v in _TH_SYMBOLS.items()])
fig.add_annotation(
    text=symbol_hint, xref="paper", yref="paper",
    x=0, y=-0.15, showarrow=False,
    font=dict(size=10, color="#64748b"), xanchor="left"
)

fig.show()

C:\Users\eieiz\AppData\Local\Temp\ipykernel_24096\1871931467.py:36: RuntimeWarning:

divide by zero encountered in divide



In [4]:
import pandas as pd
from dash import Dash, dash_table, html, dcc, Input, Output, State
from dash.dash_table.Format import Format, Scheme
import io

df['Detection'] = df['Seg_Detected'].astype(str) + " / " + df['Seg_Total'].astype(str)
percent_cols = ['Precision', 'Recall', 'F1', 'Accuracy'] + [c for c in df.columns if 'F1_NOV' in c]

# 2. ฟังก์ชัน In-cell Bar (สีเขียวอ่อน)
def data_bars(df, column):
    styles = []
    for i in range(1, 21):
        bound = i * 5
        styles.append({
            'if': {'filter_query': f'{{{column}}} >= {bound-5}', 'column_id': column},
            'background': f'linear-gradient(90deg, #dcfce7 0%, #dcfce7 {bound}%, white {bound}%, white 100%)',
            'paddingBottom': 8, 'paddingTop': 8
        })
    return styles

_MODEL_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]
model_color_map = {m: _MODEL_COLORS[i % len(_MODEL_COLORS)] for i, m in enumerate(df['Model'].unique())}

# 3. เริ่ม App (เพิ่ม html2canvas สำหรับโหลด PNG)
app = Dash(__name__, external_scripts=["https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js"])

app.layout = html.Div([
    # Header & Buttons
    html.Div([
        html.Div([
            html.H2("Model Performance Matrix", style={'fontFamily': 'sans-serif', 'margin': '0', 'fontSize': '32px'}),
            html.P("In-cell visualization with full metric exports", style={'color': '#64748b', 'fontFamily': 'sans-serif'})
        ], style={'flex': '1'}),
        
        html.Div([
            html.Button("🖼️ Download PNG", id="btn-download-png", 
                        style={'backgroundColor': '#16a34a', 'color': 'white', 'border': 'none', 
                               'padding': '12px 24px', 'borderRadius': '6px', 'cursor': 'pointer',
                               'fontWeight': 'bold', 'marginRight': '10px'}),
            html.Button("📥 Download CSV", id="btn-download-csv", 
                        style={'backgroundColor': '#2563eb', 'color': 'white', 'border': 'none', 
                               'padding': '12px 24px', 'borderRadius': '6px', 'cursor': 'pointer',
                               'fontWeight': 'bold'}),
            dcc.Download(id="download-csv"),
            html.Div(id="png-status") # สำหรับสถานะการโหลดรูป
        ], style={'display': 'flex'})
    ], style={'padding': '30px', 'borderBottom': '1px solid #e2e8f0', 'backgroundColor': 'white'}),

    # ตาราง (ใส่ id 'table-wrapper' เพื่อใช้ถ่ายรูป)
    html.Div([
        dash_table.DataTable(
            id='main-table',
            data=df.to_dict('records'),
            columns=[{
                "name": i.replace('_', ' ').upper(), "id": i, "type": "numeric" if i in percent_cols else "text",
                "format": Format(precision=1, scheme=Scheme.fixed).symbol_suffix(' %') if i in percent_cols else None
            } for i in ['Model', 'Threshold', 'Detection'] + [c for c in df.columns if c not in ['Model', 'Threshold', 'Detection', 'Seg_Detected', 'Seg_Total']]],
            sort_action="native",
            fixed_columns={'headers': True, 'data': 2},
            style_table={'minWidth': '100%'},
            style_cell={
                'textAlign': 'left', 'fontFamily': 'sans-serif', 'padding': '18px 20px', 
                'fontSize': '16px', # [ขยายฟอนต์ใหญ่มาก]
                'borderBottom': '1px solid #f1f5f9'
            },
            style_header={
                'backgroundColor': '#f8fafc', 'fontWeight': 'bold', 'color': '#475569', 
                'fontSize': '14px', 'borderBottom': '2px solid #e2e8f0'
            },
            style_data_conditional=[
                *[{
                    'if': {'filter_query': f'{{Model}} = "{m}"', 'column_id': 'Model'},
                    'color': color, 'fontWeight': 'bold', 'borderLeft': f'8px solid {color}'
                } for m, color in model_color_map.items()],
                *[style for col in percent_cols for style in data_bars(df, col)],
                {'if': {'column_id': percent_cols}, 'color': '#065f46', 'fontWeight': 'bold'}
            ],
        )
    ], id='table-container', style={'padding': '20px', 'backgroundColor': 'white'})
])

# 4. [CALLBACKS]
# ปุ่มโหลด CSV
@app.callback(Output("download-csv", "data"), Input("btn-download-csv", "n_clicks"), prevent_initial_call=True)
def download_csv(n_clicks):
    return dcc.send_data_frame(df.to_csv, "performance_results.csv", index=False)

# ปุ่มโหลด PNG (JavaScript ทำงานฝั่ง Browser)
app.clientside_callback(
    """
    function(n_clicks) {
        if(n_clicks > 0) {
            const table = document.getElementById('table-container');
            html2canvas(table, {backgroundColor: "#ffffff", scale: 2}).then(canvas => {
                const link = document.createElement('a');
                link.download = 'model_performance_summary.png';
                link.href = canvas.toDataURL("image/png");
                link.click();
            });
        }
        return "";
    }
    """,
    Output("png-status", "children"),
    Input("btn-download-png", "n_clicks"),
)

if __name__ == '__main__':
    app.run(debug=True)

In [3]:
param = pd.read_csv('last.csv').drop(columns=['rank'])
param.drop_duplicates(inplace=True)

In [4]:
param

,arch,hidden,seq_len,epochs,lr,ewma,th_name,recalc_n,precision,recall,f1,accuracy,seg_det,seg_total,score
0,GRU-AE,128,150,30,0.005,0.3,TH3 z3-10 w80 r1,1,95.5,100.0,97.6,99.2,6,6,97.6
1,GRU-AE,128,150,30,0.005,0.3,TH3 z3-10 w120 r1,1,95.3,100.0,97.5,99.0,6,6,97.5
2,GRU-AE,128,150,30,0.005,0.3,TH4 α3 w150 c2 r1,1,94.6,100.0,97.1,98.9,6,6,97.1
3,GRU-AE,128,150,30,0.005,0.3,TH4 α3 w150 c5 r1,1,94.6,100.0,97.1,98.9,6,6,97.1
4,GRU-AE,64,150,30,0.005,0.3,TH4 α3 w150 c2 r10,10,92.9,100.0,96.1,98.5,6,6,96.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,GRU-AE,256,50,30,0.005,0.3,TH2 α4.5 w80 r1,1,23.6,25.0,24.3,59.9,3,6,24.3
230,GRU-AE,64,50,30,0.005,0.3,TH3 z3-10 w80 r1,1,23.0,25.0,24.0,59.8,3,6,24.0
231,GRU-AE,64,50,30,0.005,0.3,TH2 α4.5 w80 r10,10,18.2,25.0,21.1,59.4,3,6,21.1
232,GRU-AE,64,50,30,0.005,0.3,TH2 α3.5 w80 r10,10,17.9,25.0,20.8,59.4,3,6,20.8


In [5]:
param.reset_index(inplace=True)
param['index'] = param['index'] + 1
param

,index,arch,hidden,seq_len,epochs,lr,ewma,th_name,recalc_n,precision,recall,f1,accuracy,seg_det,seg_total,score
0,1,GRU-AE,128,150,30,0.005,0.3,TH3 z3-10 w80 r1,1,95.5,100.0,97.6,99.2,6,6,97.6
1,2,GRU-AE,128,150,30,0.005,0.3,TH3 z3-10 w120 r1,1,95.3,100.0,97.5,99.0,6,6,97.5
2,3,GRU-AE,128,150,30,0.005,0.3,TH4 α3 w150 c2 r1,1,94.6,100.0,97.1,98.9,6,6,97.1
3,4,GRU-AE,128,150,30,0.005,0.3,TH4 α3 w150 c5 r1,1,94.6,100.0,97.1,98.9,6,6,97.1
4,5,GRU-AE,64,150,30,0.005,0.3,TH4 α3 w150 c2 r10,10,92.9,100.0,96.1,98.5,6,6,96.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,230,GRU-AE,256,50,30,0.005,0.3,TH2 α4.5 w80 r1,1,23.6,25.0,24.3,59.9,3,6,24.3
230,231,GRU-AE,64,50,30,0.005,0.3,TH3 z3-10 w80 r1,1,23.0,25.0,24.0,59.8,3,6,24.0
231,232,GRU-AE,64,50,30,0.005,0.3,TH2 α4.5 w80 r10,10,18.2,25.0,21.1,59.4,3,6,21.1
232,233,GRU-AE,64,50,30,0.005,0.3,TH2 α3.5 w80 r10,10,17.9,25.0,20.8,59.4,3,6,20.8


In [6]:
param['group'] = param['th_name'].str.extract(r'(TH\d+)')

# 3. เรียงลำดับ score จากมากไปน้อย และเลือกแถวแรกของแต่ละกลุ่ม
result = (param.sort_values('score', ascending=False)
            .drop_duplicates('group')
            .sort_values('group')) # เรียงกลุ่มให้ดูง่ายขึ้น

# ลบคอลัมน์ group ทิ้งถ้าไม่ต้องการใช้ต่อ
result = result.drop(columns=['group']).sort_values('score', ascending=False)

result

,index,arch,hidden,seq_len,epochs,lr,ewma,th_name,recalc_n,precision,recall,f1,accuracy,seg_det,seg_total,score
0,1,GRU-AE,128,150,30,0.005,0.3,TH3 z3-10 w80 r1,1,95.5,100.0,97.6,99.2,6,6,97.6
2,3,GRU-AE,128,150,30,0.005,0.3,TH4 α3 w150 c2 r1,1,94.6,100.0,97.1,98.9,6,6,97.1
25,26,GRU-AE,128,150,30,0.005,0.3,TH2 α3.5 w80 r10,10,90.8,100.0,94.6,98.6,6,6,94.6
82,83,GRU-AE,128,150,30,0.005,0.3,TH1 P99 w50 r30,30,81.4,100.0,87.8,95.7,6,6,87.8


In [ ]:
# # 2. หา Top 3 จากค่า F1
# top3_df = param.nlargest(3, 'f1').reset_index(drop=True)

# # 3. สลับตำแหน่งให้อันดับ 1 อยู่ตรงกลาง: [อันดับ 2, อันดับ 1, อันดับ 3]
# podium_rows = [top3_df.iloc[1], top3_df.iloc[0], top3_df.iloc[2]]

# # กำหนดสีให้อันดับ [2, 1, 3] -> แดง, น้ำเงิน, เขียว
# _COLORS = ["#dc2626", "#2563eb", "#16a34a"]

# # 4. สร้างกราฟ Podium แบบ 3 แท่ง
# fig = go.Figure()

# for i, row in enumerate(podium_rows):
#     # ปรับแต่งขนาดและสีของแท่ง
#     is_winner = (i == 1)
    
#     fig.add_trace(go.Bar(
#         x=[f"Rank {row.name + 1}"], # ชื่อแกน x ซ่อนไว้แต่ต้องมีค่า
#         y=[row['f1']],
#         marker=dict(
#             color=_COLORS[i],
#             line=dict(color='white', width=2)
#         ),
#         text=f"<b>{row['f1']:.1f}%</b>",
#         textposition='outside',
#         textfont=dict(size=22 if is_winner else 18, color=_COLORS[i]),
#         width=0.4 if is_winner else 0.35, # แท่งตรงกลางอ้วนกว่านิดนึง
#         hoverinfo='skip'
#     ))

# fig.update_layout(
#     plot_bgcolor='rgba(0,0,0,0)',
#     paper_bgcolor='rgba(0,0,0,0)',
#     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), # ซ่อนแกน x
#     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 110]), # เผื่อที่ให้ Text ด้านบน
#     margin=dict(t=40, b=0, l=0, r=0),
#     showlegend=False,
#     height=250
# )

# # 5. ฟังก์ชันสำหรับวาด Card ของแต่ละโมเดล
# def create_card(row, color, rank):
#     is_winner = (rank == 1)
#     return html.Div([
#         # ป้ายบอกอันดับ
#         html.Div(f"RANK {rank}", style={
#             'backgroundColor': color, 'color': 'white', 'padding': '6px 16px', 
#             'borderRadius': '20px', 'display': 'inline-block', 'fontSize': '12px', 
#             'fontWeight': 'bold', 'marginBottom': '10px'
#         }),
        
#         # ชื่อ Architecture และ Threshold
#         html.H3(row['arch'], style={'margin': '0 0 5px 0', 'color': '#0f172a', 'fontSize': '20px' if is_winner else '18px'}),
#         html.P(f"Threshold: {row['th_name']}", style={'margin': '0 0 15px 0', 'color': '#64748b', 'fontSize': '14px', 'fontWeight': '500'}),
        
#         # กล่องใส่ Parameter ไฮไลต์ (ดึงจากคอลัมน์ของคุณเป๊ะๆ)
#         html.Div([
#             html.Div([html.Span("Hidden", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['hidden'])]),
#             html.Div([html.Span("Seq Len", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['seq_len'])]),
#             html.Div([html.Span("Epochs", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['epochs'])]),
#             html.Div([html.Span("LR", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['lr'])]),
#             html.Div([html.Span("EWMA", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(str(row['ewma']))]),
#         ], style={'display': 'flex', 'justifyContent': 'space-between', 'backgroundColor': '#f8fafc', 
#                   'padding': '12px', 'borderRadius': '8px', 'fontSize': '13px', 'marginBottom': '15px'}),
        
#         # กล่องใส่ Metrics ย่อย
#         html.Div([
#             html.Div([
#                 html.Span("Precision", style={'color': '#64748b', 'fontSize': '12px'}),
#                 html.P(f"{row['precision']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155'})
#             ]),
#             html.Div([
#                 html.Span("Recall", style={'color': '#64748b', 'fontSize': '12px'}),
#                 html.P(f"{row['recall']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155'})
#             ]),
#             html.Div([
#                 html.Span("Accuracy", style={'color': '#64748b', 'fontSize': '12px'}),
#                 html.P(f"{row['accuracy']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155'})
#             ]),
#             html.Div([
#                 html.Span("Detected", style={'color': '#64748b', 'fontSize': '12px'}),
#                 html.P(f"{row['seg_det']}/{row['seg_total']}", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155'})
#             ]),
#         ], style={'display': 'flex', 'justifyContent': 'space-between', 'textAlign': 'center'})
        
#     ], style={
#         'flex': '1', 'backgroundColor': 'white', 'padding': '25px', 'margin': '0 15px',
#         'borderRadius': '16px', 'boxShadow': '0 10px 25px -5px rgba(0,0,0,0.1)' if is_winner else '0 4px 6px -1px rgba(0,0,0,0.05)',
#         'borderTop': f'6px solid {color}',
#         'transform': 'scale(1.05)' if is_winner else 'scale(1.0)', # เด้งขึ้นมาถ้าเป็นอันดับ 1
#         'zIndex': '10' if is_winner else '1',
#         'border': f'2px solid {color}' if is_winner else '1px solid #e2e8f0'
#     })

# # 6. ประกอบร่าง Dash App
# app = Dash(__name__)

# app.layout = html.Div([
#     html.H1("🏆 Top 3 Best Hyperparameter Configurations", style={'textAlign': 'center', 'fontFamily': 'sans-serif', 'color': '#0f172a', 'marginBottom': '5px'}),
#     html.P("Ranked by F1 Score with Architecture and Parameters details", style={'textAlign': 'center', 'color': '#64748b', 'fontFamily': 'sans-serif', 'marginBottom': '30px'}),
    
#     # 1. กราฟแท่ง 3 อันดับ
#     html.Div([
#         dcc.Graph(figure=fig, config={'displayModeBar': False})
#     ], style={'maxWidth': '1000px', 'margin': '0 auto', 'marginBottom': '-20px', 'position': 'relative', 'zIndex': '0'}),

#     # 2. การ์ด Parameter (เอามาวางทับข้างล่างกราฟนิดนึง)
#     html.Div([
#         create_card(podium_rows[0], _COLORS[0], rank=2), # ซ้าย
#         create_card(podium_rows[1], _COLORS[1], rank=1), # กลาง (ที่ 1)
#         create_card(podium_rows[2], _COLORS[2], rank=3)  # ขวา
#     ], style={'display': 'flex', 'justifyContent': 'center', 'alignItems': 'flex-start', 'maxWidth': '1200px', 'margin': '0 auto', 'position': 'relative', 'zIndex': '10', 'fontFamily': 'sans-serif'})

# ], style={'backgroundColor': '#f8fafc', 'minHeight': '100vh', 'padding': '40px 20px'})

# if __name__ == '__main__':
#     app.run(debug=True)

In [7]:
# 2. หา Top 4 จากค่า F1 (ข้อมูลจะถูกเรียงจากมากไปน้อย 1 -> 4 อัตโนมัติ)
top4_df = result.nlargest(4, 'f1').reset_index(drop=True)

# กำหนดสีตาม Rank [1, 2, 3, 4]
RANK_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706"]

# 3. สร้างกราฟแท่งเรียงซ้ายไปขวา
fig = go.Figure()

for i, row in top4_df.iterrows():
    rank = i + 1
    is_winner = (rank == 1)
    
    fig.add_trace(go.Bar(
        x=[f"Rank {rank}"],
        y=[row['f1']],
        marker=dict(
            color=RANK_COLORS[i],
            line=dict(color='white', width=2)
        ),
        text=f"<b>{row['f1']:.1f}%</b>",
        textposition='outside',
        textfont=dict(size=20 if is_winner else 16, color=RANK_COLORS[i]),
        width=0.5,
        hoverinfo='skip'
    ))

fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=False, zeroline=False, tickfont=dict(size=14, weight='bold', color='#475569')), 
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 115]), # เผื่อที่ให้ text
    margin=dict(t=40, b=30, l=0, r=0),
    showlegend=False,
    height=300
)

# 4. ฟังก์ชันวาด Card
def create_card(row, rank, color):
    is_winner = (rank == 1)
    
    return html.Div([
        # ป้ายบอกอันดับ
        html.Div(f"RANK {rank}", style={
            'backgroundColor': color, 'color': 'white', 'padding': '4px 12px', 
            'borderRadius': '20px', 'display': 'inline-block', 'fontSize': '12px', 
            'fontWeight': 'bold', 'marginBottom': '10px'
        }),
        
        # ชื่อ Architecture และ Threshold
        html.H3(row['arch'], style={'margin': '0 0 5px 0', 'color': '#0f172a', 'fontSize': '18px' if is_winner else '16px'}),
        html.P(f"TH: {row['th_name']}", style={'margin': '0 0 15px 0', 'color': '#64748b', 'fontSize': '13px', 'fontWeight': '500'}),
        
        # กล่องใส่ Parameter ไฮไลต์
        html.Div([
            html.Div([html.Span("Hid", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['hidden'])]),
            html.Div([html.Span("Seq", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['seq_len'])]),
            html.Div([html.Span("Ep", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['epochs'])]),
            html.Div([html.Span("LR", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B(row['lr'])]),
            html.Div([html.Span("EWM", style={'color': '#94a3b8', 'fontSize': '11px'}), html.Br(), html.B("Y" if row['ewma'] else "N")]),
        ], style={'display': 'flex', 'justifyContent': 'space-between', 'backgroundColor': '#f8fafc', 
                  'padding': '12px 10px', 'borderRadius': '8px', 'fontSize': '12px', 'marginBottom': '15px'}),
        
        # กล่องใส่ Metrics ย่อย
        html.Div([
            html.Div([
                html.Span("Prec", style={'color': '#64748b', 'fontSize': '11px'}),
                html.P(f"{row['precision']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155', 'fontSize': '13px'})
            ]),
            html.Div([
                html.Span("Rec", style={'color': '#64748b', 'fontSize': '11px'}),
                html.P(f"{row['recall']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155', 'fontSize': '13px'})
            ]),
            html.Div([
                html.Span("Acc", style={'color': '#64748b', 'fontSize': '11px'}),
                html.P(f"{row['accuracy']:.1f}%", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155', 'fontSize': '13px'})
            ]),
            html.Div([
                html.Span("Det", style={'color': '#64748b', 'fontSize': '11px'}),
                html.P(f"{row['seg_det']}/{row['seg_total']}", style={'margin': '0', 'fontWeight': 'bold', 'color': '#334155', 'fontSize': '13px'})
            ]),
        ], style={'display': 'flex', 'justifyContent': 'space-between', 'textAlign': 'center'})
        
    ], style={
        'flex': '1', 'backgroundColor': 'white', 'padding': '20px 15px', 'margin': '0 10px',
        'borderRadius': '16px', 
        'boxShadow': '0 10px 20px -5px rgba(0,0,0,0.15)' if is_winner else '0 4px 6px -1px rgba(0,0,0,0.05)',
        'borderTop': f'8px solid {color}' if is_winner else f'5px solid {color}',
        'transform': 'scale(1.02)' if is_winner else 'scale(1.0)', # ให้อันดับ 1 ใหญ่กว่านิดนึง
        'border': f'2px solid {color}' if is_winner else '1px solid #e2e8f0',
        'minWidth': '220px'
    })

# 5. ประกอบร่าง Dash App
app = Dash(__name__)

app.layout = html.Div([
    html.H1("🏆 Top 4 Hyperparameter Configurations", style={'textAlign': 'center', 'fontFamily': 'sans-serif', 'color': '#0f172a', 'marginBottom': '5px', 'fontSize': '28px'}),
    html.P("Ranked by F1 Score (Left to Right)", style={'textAlign': 'center', 'color': '#64748b', 'fontFamily': 'sans-serif', 'marginBottom': '20px'}),
    
    # กราฟแท่ง 4 อันดับ เรียงซ้ายไปขวา
    html.Div([
        dcc.Graph(figure=fig, config={'displayModeBar': False})
    ], style={'maxWidth': '1000px', 'margin': '0 auto', 'marginBottom': '10px'}),

    # การ์ด Parameter เรียง 4 ใบ ซ้ายไปขวา
    html.Div([
        create_card(row, i+1, RANK_COLORS[i]) for i, row in top4_df.iterrows()
    ], style={'display': 'flex', 'justifyContent': 'center', 'maxWidth': '1200px', 'margin': '0 auto', 'fontFamily': 'sans-serif', 'flexWrap': 'wrap'})

], style={'backgroundColor': '#f8fafc', 'minHeight': '100vh', 'padding': '30px 20px'})

if __name__ == '__main__':
    app.run(debug=True)

In [8]:
th1 = 33 + 94 + 72 + 52+ 98 + 95 + 92
th2 = 59 + 100 + 94 + 93 + 98 + 98 + 96
th3 = 74 + 100 + 94  +93 + 98 + 99 + 98
th4 = 61 + 100 + 99 + 93 + 98 + 96 + 96

In [15]:
round(th1/7, 2), round(th2/7, 2), round(th3/7, 2), round(th4/7, 2)

(76.57, 91.14, 93.71, 91.86)

In [14]:
import pandas as pd

data = {
    "Run_ID": [
        "NOV_17_1",
        "NOV_20_4",
        "NOV_26_5",
        "NOV_26_12",
        "NOV_27_3",
        "NOV_27_6",
        "NOV_28_3",
        "Mean F1"
    ],
    "Threshold 1": [33, 94, 72, 52, 98, 95, 92, 77],
    "Threshold 2": [59, 100, 94, 93, 98, 98, 96, 91],
    "Threshold 3": [74, 100, 94, 93, 98, 99, 98, 94],
    "Threshold 4": [61, 100, 99, 93, 98, 96, 96, 92],
}

df = pd.DataFrame(data)

# ถ้าต้องการให้เป็น %
df_percent = df.copy()
for col in df.columns[1:]:
    df_percent[col] = df_percent[col].astype(str) + "%"

print(df)
df_percent

      Run_ID  Threshold 1  Threshold 2  Threshold 3  Threshold 4
0   NOV_17_1           33           59           74           61
1   NOV_20_4           94          100          100          100
2   NOV_26_5           72           94           94           99
3  NOV_26_12           52           93           93           93
4   NOV_27_3           98           98           98           98
5   NOV_27_6           95           98           99           96
6   NOV_28_3           92           96           98           96
7    Mean F1           77           91           94           92


,Run_ID,Threshold 1,Threshold 2,Threshold 3,Threshold 4
0,NOV_17_1,33%,59%,74%,61%
1,NOV_20_4,94%,100%,100%,100%
2,NOV_26_5,72%,94%,94%,99%
3,NOV_26_12,52%,93%,93%,93%
4,NOV_27_3,98%,98%,98%,98%
5,NOV_27_6,95%,98%,99%,96%
6,NOV_28_3,92%,96%,98%,96%
7,Mean F1,77%,91%,94%,92%


In [16]:
import pandas as pd

data = {
    "Run_ID": ["Validation set", "Testing set"],
    "Threshold 1": [87.8, 76.6],
    "Threshold 2": [94.6, 91.1],
    "Threshold 3": [97.6, 93.7],
    "Threshold 4": [97.1, 91.9],
}

df = pd.DataFrame(data).set_index("Run_ID")

# คำนวณ Gap
gap = df.loc["Validation set"] - df.loc["Testing set"]

# เพิ่มแถว Gap
df.loc["Gap"] = gap

df

,Threshold 1,Threshold 2,Threshold 3,Threshold 4
Run_ID,,,,
Validation set,87.8,94.6,97.6,97.1
Testing set,76.6,91.1,93.7,91.9
Gap,11.2,3.5,3.9,5.2


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 1. ข้อมูลของคุณ
data = {
    "Run_ID": ["Validation set", "Testing set"],
    "Percentile": [87.8, 76.6],
    "Sliding μ + α·σ": [94.6, 91.1],
    "Adaptive-z": [97.6, 93.7],
    "Entropy-lock": [97.1, 91.9],
}

df = pd.DataFrame(data).set_index("Run_ID")

# แปลงข้อมูลให้อยู่ในรูปแบบที่พล็อตง่าย (Melt / Transpose)
# แกน X จะเป็น Threshold, แกน Y จะเป็นคะแนน
df_plot = df.T.reset_index()
df_plot.columns = ['Threshold', 'Validation', 'Testing']
df_plot['Gap'] = df_plot['Validation'] - df_plot['Testing']

# 2. ตั้งค่าธีมสี
COLOR_VAL = "#2563eb"   # สีน้ำเงิน (Validation)
COLOR_TEST = "#f97316"  # สีส้ม (Testing)
COLOR_GAP = "#94a3b8"   # สีเทา (เส้นโยง Gap)

fig = go.Figure()

# 3. สร้างเส้นโยงความห่าง (Gap Lines) ก่อน เพื่อให้อยู่ด้านหลังสุด
for i, row in df_plot.iterrows():
    # วาดเส้นประแนวตั้ง
    fig.add_shape(
        type="line",
        x0=row['Threshold'], y0=row['Validation'],
        x1=row['Threshold'], y1=row['Testing'],
        line=dict(color=COLOR_GAP, width=2, dash="dot"),
        layer="below"
    )
    
    # วาดป้ายข้อความบอกระยะ Gap ตรงกึ่งกลางเส้น
    mid_y = (row['Validation'] + row['Testing']) / 2
    fig.add_annotation(
        x=row['Threshold'], 
        y=mid_y,
        text=f"<b>-{row['Gap']:.1f}</b>", # โชว์ค่า Gap ติดลบ (drop ลงมา)
        showarrow=False,
        font=dict(size=12, color="#ef4444"), # ใช้สีแดงสื่อถึงคะแนนที่ตกลง
        bgcolor="white", # พื้นหลังสีขาวให้ทับเส้นประ
        bordercolor="#e2e8f0",
        borderwidth=1,
        borderpad=4,
    )

# 4. วาดเส้น Validation Set (อยู่ด้านบน)
fig.add_trace(go.Scatter(
    x=df_plot['Threshold'],
    y=df_plot['Validation'],
    mode="lines+markers+text",
    name="Validation set",
    line=dict(color=COLOR_VAL, width=3),
    marker=dict(size=12, color="white", line=dict(color=COLOR_VAL, width=3)),
    text=[f"{v:.1f}" for v in df_plot['Validation']],
    textposition="top center",
    textfont=dict(size=14, color=COLOR_VAL)
))

# 5. วาดเส้น Testing Set (อยู่ด้านล่าง)
fig.add_trace(go.Scatter(
    x=df_plot['Threshold'],
    y=df_plot['Testing'],
    mode="lines+markers+text",
    name="Testing set",
    line=dict(color=COLOR_TEST, width=3),
    marker=dict(size=12, color="white", line=dict(color=COLOR_TEST, width=3)),
    text=[f"{v:.1f}" for v in df_plot['Testing']],
    textposition="bottom center",
    textfont=dict(size=14, color=COLOR_TEST)
))

# 6. ปรับแต่ง Layout ให้สะอาดตา (Dashboard Style)
fig.update_layout(
    title=dict(
        text="<b>Performance Gap: Validation vs Testing</b><br><span style='font-size:13px; color:#64748b;'>Comparing F1-scores across different thresholds</span>",
        font=dict(size=20, color="#0f172a"),
        x=0.05
    ),
    xaxis=dict(
        title="",
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=14, color="#475569", weight="bold")
    ),
    yaxis=dict(
        title="Mean F1-Score (%)",
        showgrid=True,
        gridcolor="#f1f5f9",
        zeroline=False,
        range=[70, 105], # เผื่อพื้นที่ด้านบน-ล่างให้ Text
        tickfont=dict(size=12, color="#64748b")
    ),
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
        font=dict(size=13)
    ),
    margin=dict(l=60, r=40, t=100, b=50),
    height=500,width=750
)

# แสดงผลกราฟ
fig.show()